## Notebook Purpose

This notebook builds a **day ↔ night image translation pipeline** using `img2img-turbo` (CycleGAN-style training and inference):

1. Downloads a day/night city-view dataset.
2. Splits images into train/test sets in the expected folder format.
3. Defines fixed domain prompts (A = night, B = day).
4. Trains a translation model.
5. Runs inference on a sample input image.

---

## What Each Step Does

1. **Clone repository**
    - Clones `img2img-turbo` source code.

2. **Dataset section header**
    - Markdown title: *Cityview Dataset*.

3. **Download + unzip dataset**
    - Downloads Kaggle dataset zip into `datasets/`.
    - Extracts to `datasets/`.

4. **Prepare utility function**
    - Defines `copy_images(source_dir, dest_dir, a_or_b)`:
      - Collects images from a source folder.
      - Splits into train/test (80/20).
      - Copies files into:
         - `daynight/train_A`, `daynight/test_A`
         - `daynight/train_B`, `daynight/test_B`

5. **Create dataset structure**
    - Maps:
      - `datasets/data/0` → domain `A`
      - `datasets/data/1` → domain `B`

6. **Create fixed prompts**
    - Writes:
      - `daynight/fixed_prompt_a.txt`: “a photo taken at night”
      - `daynight/fixed_prompt_b.txt`: “a photo taken during the day”

7. **Train model**
    - Launches `train_cyclegan_turbo.py` with dataset `./daynight`.
    - Saves training outputs to `./daynight_trained`.

8. **Run inference**
    - Runs unpaired inference (`b2a`) on `images/night.jpg`.
    - Writes output image(s) to `output/`.

---

## How to Run

1. Ensure environment has:
    - Python, PyTorch (CUDA recommended), `accelerate`, `xformers`, `scikit-learn`, `aria2`, `unzip`.
2. Run cells in order from top to bottom.
3. Verify key paths exist before training/inference:
    - Dataset extracted at `datasets/data/0` and `datasets/data/1`.
    - Input image exists at `images/night.jpg`.
4. After training, use the produced checkpoint path for inference (update `--model_path` if needed).  
    - The current inference cell uses `checkpoints/night2day.pkl`, which must exist or be replaced with your trained checkpoint path.

---

## Expected Outputs

- Prepared dataset folders under `daynight/`.
- Training artifacts under `daynight_trained/`.
- Generated translated image(s) under `output/`.

In [ ]:
!git clone https://github.com/GaParmar/img2img-turbo

## Cityview Dataset

In [ ]:
!aria2c -x4 https://www.kaggle.com/api/v1/datasets/download/sohamndeshmukh/day-night-city-view-dataset --auto-file-renaming=false --allow-overwrite=false -o datasets/day-night-city-view.zip
!unzip datasets/day-night-city-view.zip -d datasets

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

def copy_images(source_dir, dest_dir, a_or_b):
  # Define paths
  train_day_dest_dir = os.path.join(dest_dir, f"train_{a_or_b}")
  test_day_dest_dir = os.path.join(dest_dir, f"test_{a_or_b}")

  os.makedirs(train_day_dest_dir, exist_ok=True)
  os.makedirs(test_day_dest_dir, exist_ok=True)

  # List all image files
  all_day_images = []
  if os.path.exists(source_dir):
      for root, _, files in os.walk(source_dir):
          for file in files:
              if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                  all_day_images.append(os.path.join(root, file))

  print(f"Found {len(all_day_images)} images.")

  if all_day_images:
      # Split into training and testing sets
      train_day_images, test_day_images = train_test_split(
          all_day_images, test_size=0.2, random_state=42
      )

      print(f"Splitting images: {len(train_day_images)} for training, {len(test_day_images)} for testing.")

      # Copy training images
      for img_path in train_day_images:
          shutil.copy2(img_path, train_day_dest_dir)
      print(f"Copied {len(train_day_images)} training images to {train_day_dest_dir}")

      # Copy testing images
      for img_path in test_day_images:
          shutil.copy2(img_path, test_day_dest_dir)
      print(f"Copied {len(test_day_images)} testing images to {test_day_dest_dir}")
  else:
      print("No day images found to process.")





In [ ]:
copy_images("datasets/data/0", "daynight", "A")
copy_images("datasets/data/1", "daynight", "B")

In [ ]:
!echo "a photo taken at night" > daynight/fixed_prompt_a.txt
!echo "a photo taken during the day" > daynight/fixed_prompt_b.txt

In [ ]:
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!accelerate launch img2img-turbo/src/train_cyclegan_turbo.py \
  --dataset_folder ./daynight \
  --output_dir ./daynight_trained \
  --train_batch_size 1 \
  --train_img_prep resize_256 \
  --val_img_prep resize_256 \
  --max_train_steps 15000 \
  --tracker_project_name oxford_night2day \
  --enable_xformers_memory_efficient_attention

In [ ]:
!python img2img-turbo/src/inference_unpaired.py --direction b2a --model_path checkpoints/night2day.pkl --prompt "a city in the day" --input_image images/night.jpg --output_dir output --use_fp16